如果将状态检查点(checkpointer)保存在内存，进程结束则状态丢失，生产环境不可接受。因此，生产环境要用持久化的外部存储介质，如PostgreSQL。LangGraph提供的checkpointer后端列表如下
https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries

此处选择PostgreSQL作为持久化器。

数据库环境准备

在此之前，先准备好PostgreSQL环境，此处在云服务器的Ubuntu系统安装PostgreSQL。大家需要根据自己云服务器位置，更改URL中的IP即可。

具体云服务器安装和PostgreSQL的安装，见文件资料下的《Linux云服务器安装与PostgreSQL安装》

说明：

1、上述URL中的用户名、密码、IP地址，需要根据自己的情况替换。

2、要在LangChain中对接PostgreSQL，还需要额外的依赖，比如：

langgraph-checkpoint-postgres



## 创建的数据表


1. **checkpoints**：这是主表，存每个thread在某个时刻的checkpoint快照。
2. **checkpoint_blobs**：这张表专门存不适合直接内联进`checkpoints.checkpoint`的较复杂channel值。
3. **checkpoint_writes**：这张表存的是中间写入 / pending writes，不是最终完整checkpoint。
4. **checkpoint_migrations**：这张表不是业务数据表，而是迁移版本表。

## 实例代码

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

In [6]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.postgres import PostgresSaver
import os
from dotenv import load_dotenv
load_dotenv(override=True)

DB_URL = os.getenv("DATABASE_URL")
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # 初始化PostgreSQL数据库
    checkpointer.setup()

    agent = create_agent(
        model=model,
        checkpointer=checkpointer
    )

    config = {"configurable": {"thread_id": "1"}}
    response1 = agent.invoke(
        {"messages": [HumanMessage("你好，我是老王")]},
        config=config
    )
    print("=" * 30, "-> 第一次调用 <-", "=" * 30)
    for msg in response1["messages"]:
        msg.pretty_print()

    response2 = agent.invoke(
        {"messages": [HumanMessage("你好，我是谁？")]},
        config=config
    )

    print("=" * 30, "-> 第二次调用 <-", "=" * 30)
    for msg in response2["messages"]:
        msg.pretty_print()

============================== -> 第一次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。有什么我可以帮你的？
============================== -> 第二次调用 <- ==============================
================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

你好，老王！很高兴认识你。有什么我可以帮你的？
================================ Human Message =================================

你好，我是谁？
================================== Ai Message ==================================

你好，你是老王。
